<a href="https://colab.research.google.com/github/ishanallasanagala/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, tell you honestly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [63]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    for attempt in range(tries):
      try:
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
        r = requests.get(BASE, params = params, timeout = 15)
        if r.status_code == 200:
          return pd.DataFrame(r.json()['daily'])
    # TODO:   raise on a status that is not in RETRY_STATUSES
        if r.status_code not in RETRY_STATUSES:
          raise RuntimeError(f"Status {r.status_code}: {r.text[:200]}")
      except requests.RequestException:
        pass
    # TODO:   otherwise sleep (increasing each time) and try again
      time.sleep(attempt)
    # TODO: raise if every attempt failed
    raise RuntimeError(f"All {tries} attempts failed")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-10**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [64]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False
#wx for weather data
try:
  wx_raw = get_weather('2026-09-05', '2026-10-10')
  wx = wx_raw[wx_raw['time'].isin(['2026-09-05', '2026-10-10'])].copy()

except Exception as e:
  print(f'Error: {e}')
  used_fallback = True

### Part 3 — the fallback, done honestly

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [65]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-10'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
if used_fallback:
  wx = FALLBACK.copy()
  used_fallback = True
  print('using fallback')
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
wx['source'] = 'open-meteo' if not used_fallback else 'fallback sample'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [66]:
# TODO
wx = wx.rename(columns = {
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})

wx['date'] = pd.to_datetime(wx['date'])
#Retains these columns and resets index numbers
wx = wx[['date', 'temp_max', 'precip_mm', 'source']].reset_index(drop = True)


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [67]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [68]:
# TODO
if wx.loc[0, 'temp_max'] == wx.loc[1, 'temp_max']:
  print('both dates had the same amount of precipitation')
else:
  wettest_date = '2026-09-05' if wx.loc[0, 'temp_max'] > wx.loc[1, 'temp_max'] else '2026-10-10'
  print(wettest_date, 'was the date with more precipitation.')

2026-09-05 was the date with more precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [69]:
# TODO
from datetime import datetime

print('Endpoint', BASE)
print('Coordinates', (38.03, -78.51))
print('Date range', (datetime(2026, 9, 5), datetime(2026, 10, 10)))
print('Pull time', datetime.now())
print('Data source', 'fallback sample' if used_fallback else wx['source'][0])

Endpoint https://archive-api.open-meteo.com/v1/archive
Coordinates (38.03, -78.51)
Date range (datetime.datetime(2026, 9, 5, 0, 0), datetime.datetime(2026, 10, 10, 0, 0))
Pull time 2026-10-12 00:42:01.959895
Data source open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame on failure instead of raising an error, its values would either be dropped or filled with NaN depending on the merge type. From there, the chart would depict values corresponding to a situation with no rainfall where it never rained. The chart would then be interpreted as though these erroneous cells were dry days without precipitation. Nobody would catch this because there would be no error to alert users, rather there would be an edit to data. While the results would be altered at a scale ranging from minor to major, the lack of an error alert would make it impossible to be alerted directly.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [70]:
# stretch
try:
  orl_raw = get_weather('2026-09-05', '2026-10-10', lat = 28.54, lon = -81.38)
  orl = orl_raw[orl_raw['time'].isin(['2026-09-05', '2026-10-10'])].copy()

  orl = orl.rename(columns = {
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
    })

  orl['date'] = pd.to_datetime(orl['date'])
  orl['source'] = 'open-meteo'

  #Add city columns to distinguish between the locations
  orl['city'] = 'Orlando'
  wx['city'] = 'Charlottesville'

  frame = pd.concat([wx, orl], ignore_index = True)
  print(frame)

except Exception as e:
  print(f'Error: {e}')

        date  temp_max  precip_mm      source             city
0 2026-09-05      29.4        2.5  open-meteo  Charlottesville
1 2026-10-10      19.0        2.4  open-meteo  Charlottesville
2 2026-09-05      32.1        3.9  open-meteo          Orlando
3 2026-10-10      29.7       10.4  open-meteo          Orlando
